# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. We interact with a Croissant schema defining a multi-table dataset describing clinical and molecular variables for second primary colorectal cancer survivors.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure 'mlcroissant' library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load the FAIR² dataset metadata and its records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as a dict!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available record sets, their fields, and the column/field `@id`s in the Croissant schema.

In [ ]:
# List all record sets in the dataset by @id and name and print included fields by @id/name.
record_sets = dataset.record_sets
print("Record Sets (@id and name):")
for rs in record_sets:
    print(f"- @{chr(105)}d: {rs.id}, name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - @{chr(105)}d: {fld.id}, name: {getattr(fld,'name', 'N/A')}")
    print('')

## 3. Data Extraction

Load all record sets into pandas DataFrames using their `@id`s. For this dataset, we'll extract the tabular data and show available fields.

In [ ]:
# Build a mapping of record set ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set @id='{rs_id}' with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for record set @id='{rs_id}'.")

## 4. Exploratory Data Analysis (EDA)

Let's perform exploratory data analysis on the main tabular record set—filter, normalize, group. All operations are performed referencing field `@id`s.

In [ ]:
# Choose the main record set for clinical tabular data.
# (Replace this with the actual @id after inspecting the output above)
main_rs_id = None
for rs_id, df in dataframes.items():
    if df.shape[0] == 77:  # The dataset mentions N=77
        main_rs_id = rs_id
        break
assert main_rs_id is not None, "Main record set with tabular data not found."

# Print columns with their field @ids
print(f"Fields in main record set (@id='{main_rs_id}'):")
print(dataframes[main_rs_id].columns.tolist())

# For demonstration, pick an age or interval column for numeric EDA.
# We'll try typical numeric fields, adjust as appropriate.
# Let's choose a field name with e.g., 'age' or 'interval' (adjust as per real column names)
numeric_field_candidates = [col for col in dataframes[main_rs_id].columns if any(s in col.lower() for s in ['age','interval','months','years','duration'])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # e.g. '@id' of field: age_at_diagnosis, or diagnosis_interval_months
else:
    numeric_field_id = dataframes[main_rs_id].columns[0]  # Just use first column if not found
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Set a threshold for filtering, e.g., age > 50 or interval > 12
# Use median as threshold for demonstration
df = dataframes[main_rs_id]
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = float(df[numeric_field_id].median())
else:
    # Try to convert to numeric if possible (coerce errors)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = float(df[numeric_field_id].median())

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in '{numeric_field_id}' > {threshold:.1f}:")
display(filtered_df.head(3))

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a potential categorical/clinical grouping
# Choose a column with string/object type for grouping, e.g., 'sex', 'anatomical_location', etc.
cat_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
if cat_field_candidates:
    group_field = cat_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean').reset_index()
    print(f"Grouped average of '{numeric_field_id}' by category @id='{group_field}':")
    display(grouped_df)
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of our selected numeric field and its relation to a grouping category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, plot boxplot
if 'group_field' in locals():
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion

- Loaded and explored the FAIR² clinical dataset using the Croissant schema and `mlcroissant`.
- Parsed available record sets with all fields referenced by their `@id`.
- Demonstrated basic exploratory data analysis and visualization using unique schema identifiers.
- For further scientific exploration, refer to the full Croissant schema for advanced data interoperability and machine learning workflows in clinical settings.